# Titanic
## Score: .76076

In [23]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    VotingClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()
DATA = ROOT / "titanic"
assert (DATA / "train.csv").exists()

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [24]:
def extract_title(name: str) -> str:
    m = re.search(r",\s*([^\.]+)\.", name)
    return m.group(1).strip() if m else "Unknown"


def cabin_deck(cabin) -> str:
    if pd.isna(cabin) or not str(cabin).strip():
        return "U"
    c = str(cabin).strip()[0]
    return c if c.isalpha() else "U"


def add_features(
    df: pd.DataFrame,
    ticket_vc: pd.Series | None = None,
    surname_vc: pd.Series | None = None,
) -> pd.DataFrame:
    out = df.copy()
    out["Title"] = out["Name"].map(extract_title)
    rare = {
        "Lady",
        "Countess",
        "Capt",
        "Col",
        "Don",
        "Dr",
        "Major",
        "Rev",
        "Sir",
        "Jonkheer",
        "Dona",
    }
    out["Title"] = out["Title"].replace(
        {
            "Mlle": "Miss",
            "Ms": "Miss",
            "Mme": "Mrs",
        }
    )
    out.loc[out["Title"].isin(rare), "Title"] = "Rare"
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    out["HasCabin"] = out["Cabin"].notna().astype(int)
    out["Deck"] = out["Cabin"].map(cabin_deck)
    out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
    out["LogFare"] = np.log1p(out["Fare"])
    out["IsChild"] = np.where(out["Age"].notna(), (out["Age"] < 16).astype(int), 0)
    out["AgePclass"] = out["Age"] * out["Pclass"]
    out["FarePclass"] = out["Fare"] * out["Pclass"]
    sn = out["Name"].str.split(",").str[0].str.strip()
    if surname_vc is None:
        out["SurnameGroupSize"] = sn.map(sn.value_counts())
    else:
        out["SurnameGroupSize"] = sn.map(surname_vc).fillna(1).astype(int)
    if ticket_vc is None:
        out["TicketGroupSize"] = out.groupby("Ticket")["Ticket"].transform("count")
    else:
        out["TicketGroupSize"] = out["Ticket"].map(ticket_vc).fillna(1).astype(int)
    return out

In [25]:
def build_preprocessor() -> ColumnTransformer:
    numeric = [
        "Pclass",
        "Age",
        "SibSp",
        "Parch",
        "Fare",
        "FamilySize",
        "IsAlone",
        "HasCabin",
        "FarePerPerson",
        "LogFare",
        "IsChild",
        "TicketGroupSize",
        "SurnameGroupSize",
        "AgePclass",
        "FarePclass",
    ]
    categorical = ["Sex", "Embarked", "Title", "Deck"]
    cat_pipe = Pipeline(
        [
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        [
            ("num", SimpleImputer(strategy="median"), numeric),
            ("cat", cat_pipe, categorical),
        ]
    )


def build_model() -> VotingClassifier:
    rf = RandomForestClassifier(
        n_estimators=1200,
        max_depth=22,
        min_samples_leaf=1,
        min_samples_split=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=1,
    )
    et = ExtraTreesClassifier(
        n_estimators=1200,
        max_depth=22,
        min_samples_leaf=1,
        min_samples_split=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=1,
    )
    hgb = HistGradientBoostingClassifier(
        max_iter=600,
        learning_rate=0.04,
        max_depth=12,
        min_samples_leaf=4,
        l2_regularization=1e-3,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30,
        random_state=42,
    )
    return VotingClassifier(
        estimators=[
            ("rf", Pipeline([("prep", build_preprocessor()), ("m", rf)])),
            ("et", Pipeline([("prep", build_preprocessor()), ("m", et)])),
            ("hgb", Pipeline([("prep", build_preprocessor()), ("m", hgb)])),
        ],
        voting="soft",
        n_jobs=-1,
    )

In [26]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")

ticket_vc = train["Ticket"].value_counts()
surname_vc = train["Name"].str.split(",").str[0].str.strip().value_counts()
train_x = add_features(train.drop(columns=["Survived"]), ticket_vc, surname_vc)
train_y = train["Survived"]
test_x = add_features(test, ticket_vc, surname_vc)

model = build_model()
cv_scores = cross_val_score(model, train_x, train_y, cv=CV, scoring="accuracy", n_jobs=-1)
print(cv_scores.mean(), cv_scores.std())

oof_proba_cv = cross_val_predict(model, train_x, train_y, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
thresh_grid = np.linspace(0.22, 0.78, 113)
best_t, best_oob = 0.5, -1.0
for t in thresh_grid:
    acc = accuracy_score(train_y, (oof_proba_cv >= t).astype(int))
    if acc > best_oob:
        best_oob, best_t = acc, t
print(best_oob, best_t)

model.fit(train_x, train_y)
test_proba = model.predict_proba(test_x)[:, 1]
pred = (test_proba >= best_t).astype(int)
sub = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": pred.astype(int)})
out_path = ROOT / "submission.csv"
sub.to_csv(out_path, index=False)
print(out_path, len(sub))
sub.head(10)

0.8192768815516918 0.019156156906629064
0.8294051627384961 0.73
c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv 418


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
5,897,0
6,898,0
7,899,0
8,900,1
9,901,0


In [27]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import cross_val_predict

oof_proba = cross_val_predict(model, train_x, train_y, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
thresh_grid = np.linspace(0.22, 0.78, 113)
best_t, best_a = 0.5, -1.0
for t in thresh_grid:
    a = accuracy_score(train_y, (oof_proba >= t).astype(int))
    if a > best_a:
        best_a, best_t = a, t
oof = (oof_proba >= best_t).astype(int)

print("acc0.5", accuracy_score(train_y, (oof_proba >= 0.5).astype(int)))
print("acc_t", best_a, "t", best_t)
print(classification_report(train_y, oof, digits=4))
print(confusion_matrix(train_y, oof))
print("auc", roc_auc_score(train_y, oof_proba))

fold_acc = []
for _, te_idx in CV.split(train_x, train_y):
    fold_acc.append(accuracy_score(train_y.iloc[te_idx], oof[te_idx]))
print("folds", [round(x, 4) for x in fold_acc], "std", round(float(np.std(fold_acc)), 4))

train_aug = train.assign(bad=oof != train_y.values)
print("err sex", train_aug.groupby("Sex")["bad"].mean().round(4).to_dict())
print("err pcl", train_aug.groupby("Pclass")["bad"].mean().round(4).to_dict())
print("maj", accuracy_score(train_y, np.full(len(train_y), int(train_y.mode().iloc[0]))))

acc0.5 0.819304152637486
acc_t 0.8294051627384961 t 0.73
              precision    recall  f1-score   support

           0     0.8049    0.9545    0.8733       549
           1     0.8958    0.6287    0.7388       342

    accuracy                         0.8294       891
   macro avg     0.8504    0.7916    0.8061       891
weighted avg     0.8398    0.8294    0.8217       891

[[524  25]
 [127 215]]
auc 0.8764579938005305
folds [0.8324, 0.8258, 0.8146, 0.8427, 0.8315] std 0.0092
err sex {'female': 0.207, 'male': 0.1508}
err pcl {1: 0.2222, 2: 0.0978, 3: 0.1752}
maj 0.6161616161616161
